# Step 2: Data Cleaning & Star Schema Transformation

**E-Commerce 360° Business Intelligence Platform**

ETL pipeline producing curated dimension and fact tables.


### 1. Overview & Setup
Clean raw data, map geographical macro-regions, standardize categories, handle missing values, and construct an analytical Star Schema.

In [1]:
import os
import pandas as pd
import numpy as np

RAW_DIR = r'data/raw'
PROCESSED_DIR = r'data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)
print('Source and target directories verified.')

Source and target directories verified.


### 2. Dimension Tables Transformation
Construct `dim_customers`, `dim_products`, `dim_sellers`, and `dim_date`.

In [2]:
STATE_REGION_MAP = {
    'SP': 'Southeast', 'RJ': 'Southeast', 'MG': 'Southeast', 'ES': 'Southeast',
    'PR': 'South', 'SC': 'South', 'RS': 'South',
    'BA': 'Northeast', 'PE': 'Northeast', 'CE': 'Northeast', 'MA': 'Northeast',
    'PB': 'Northeast', 'RN': 'Northeast', 'AL': 'Northeast', 'SE': 'Northeast', 'PI': 'Northeast',
    'MT': 'Central-West', 'MS': 'Central-West', 'GO': 'Central-West', 'DF': 'Central-West',
    'AM': 'North', 'PA': 'North', 'RO': 'North', 'TO': 'North', 'AC': 'North', 'AP': 'North', 'RR': 'North'
}

# dim_customers
df_cust = pd.read_csv(f'{RAW_DIR}/olist_customers_dataset.csv')
df_cust['customer_city'] = df_cust['customer_city'].str.strip().str.title()
df_cust['customer_state'] = df_cust['customer_state'].str.strip().str.upper()
df_cust['region'] = df_cust['customer_state'].map(STATE_REGION_MAP).fillna('Other')
dim_cust = df_cust.rename(columns={'customer_zip_code_prefix': 'zip_code_prefix', 'customer_city': 'city', 'customer_state': 'state'})
dim_cust.to_csv(f'{PROCESSED_DIR}/dim_customers.csv', index=False)
print(f'dim_customers created: {dim_cust.shape}')

# dim_products
df_prod = pd.read_csv(f'{RAW_DIR}/olist_products_dataset.csv')
df_trans = pd.read_csv(f'{RAW_DIR}/product_category_name_translation.csv')
df_prod = df_prod.merge(df_trans, on='product_category_name', how='left')
df_prod['category_name_en'] = df_prod['product_category_name_english'].fillna(
    df_prod['product_category_name'].str.replace('_', ' ').str.title()
).fillna('Unknown')
df_prod['category_name_pt'] = df_prod['product_category_name'].fillna('desconhecido')
df_prod['volume_cm3'] = (df_prod['product_length_cm'].fillna(0) * df_prod['product_height_cm'].fillna(0) * df_prod['product_width_cm'].fillna(0)).round(2)
dim_prod = df_prod[['product_id', 'category_name_pt', 'category_name_en', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'volume_cm3']]
dim_prod.to_csv(f'{PROCESSED_DIR}/dim_products.csv', index=False)
print(f'dim_products created: {dim_prod.shape}')

dim_customers created: (99441, 6)


dim_products created: (32951, 11)


### 3. Fact Tables Transformation & Verification
Transform `fact_orders`, `fact_order_items`, `fact_payments`, and `fact_reviews`.

In [3]:
for name in ['dim_customers', 'dim_products', 'dim_sellers', 'dim_date', 'fact_orders', 'fact_order_items', 'fact_payments', 'fact_reviews', 'order_summary_bi']:
    path = f'{PROCESSED_DIR}/{name}.csv'
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'{name:<20}: {df.shape[0]:>8,} rows, {df.shape[1]:>2} cols')
    else:
        print(f'{name:<20}: Pending generation')

dim_customers       :   99,441 rows,  6 cols


dim_products        :   32,951 rows, 11 cols
dim_sellers         :    3,095 rows,  5 cols
dim_date            :    1,096 rows, 12 cols


fact_orders         :   99,441 rows, 12 cols


fact_order_items    :  112,650 rows,  9 cols
fact_payments       :  103,886 rows,  5 cols


fact_reviews        :   99,224 rows,  8 cols


order_summary_bi    :   99,441 rows, 27 cols
